### Classical ML

- [Logistic Regression](#Logistic-Regression)
- [Linear Regression](#Linear-Regression)
- [SVM](#SVM)
- [KNN](#KNN)
- [Naive Bayes](#Naive-Bayes)

**1. Logistic Regression**

Plain terms: start with a random guess for the weights. For every point, check how wrong that guess is, then nudge the weights a small step in the direction that makes the guess less wrong. Repeat until the wrongness stops shrinking. Sigmoid turns the raw guess into a probability; gradient descent is just this repeated small-correction process.

1. Probability, odds, log-odds: $p=\text{successes}/\text{total}\in[0,1]$. Odds $=p/(1-p)\in[0,\infty)$. Log-odds $=\ln(\text{odds})\in(-\infty,\infty)$.

| state | p | odds | log-odds |
|---|---|---|---|
| impossible | 0.0 | 0 | $-\infty$ |
| unlikely | 0.2 | 0.25 | -1.386 |
| coin toss | 0.5 | 1 | 0.0 |
| likely | 0.8 | 4.0 | +1.386 |
| certain | 1.0 | $\infty$ | $+\infty$ |


2. Why log-odds, not $p$, as the linear target: $z=w^Tx+b$ is unconstrained in $(-\infty,\infty)$; setting $p=z$ directly breaks the $[0,1]$ bound. Setting $\ln(p/(1-p))=z$ keeps domains matched. $w_j$ = change in log-odds per unit $x_j$; $e^{w_j}$ = odds ratio.

3. Sigmoid, derived from log-odds: $p/(1-p)=e^z \Rightarrow p=e^z(1-p) \Rightarrow p+pe^z=e^z \Rightarrow p=\frac{e^z}{1+e^z}=\frac{1}{1+e^{-z}}$. Squashes any real $z$ into $(0,1)$: at $z=0$, $p=0.5$; as $z\to\pm\infty$, $p\to1$ or $0$. Symmetry $\sigma(-z)=1-\sigma(z)$; derivative $\sigma'(z)=\sigma(z)(1-\sigma(z))=p(1-p)$.

4. MLE, single point: $P(y_i|x_i)=p_i^{y_i}(1-p_i)^{1-y_i}$, collapses to $p_i$ when $y_i=1$, $(1-p_i)$ when $y_i=0$. Joint likelihood (i.i.d.): $L(w)=\prod p_i^{y_i}(1-p_i)^{1-y_i}$. Probability fixes $w$, varies $y$ ("how likely are these outcomes"); likelihood fixes $(x,y)$, varies $w$ ("how plausible are these weights").

5. Log-loss, from MLE: log turns the product into a sum, avoids underflow: $\ell(w)=\sum[y_i\ln p_i+(1-y_i)\ln(1-p_i)]$. Gradient descent minimizes, MLE maximizes, negate and average: $L(w)=-\frac{1}{N}\sum[y_i\ln p_i+(1-y_i)\ln(1-p_i)]$.

6. Gradient, chain rule: $\frac{\partial L_i}{\partial w_j}=\frac{\partial L_i}{\partial p_i}\cdot\frac{\partial p_i}{\partial z_i}\cdot\frac{\partial z_i}{\partial w_j}=\left[\frac{p_i-y_i}{p_i(1-p_i)}\right]\cdot[p_i(1-p_i)]\cdot x_{ij}=(p_i-y_i)x_{ij}$. The $p(1-p)$ terms cancel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

n = 200
X0 = rng.normal(loc=[-2, -2], scale=1.0, size=(n, 2))
X1 = rng.normal(loc=[2, 2], scale=1.0, size=(n, 2))
X = np.vstack([X0, X1])
y = np.hstack([np.zeros(n), np.ones(n)])

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", alpha=0.6)
plt.title("toy data")
plt.show()

# squashes any real z into (0,1); derivative = p(1-p) -> steep at 0.5, flat at extremes
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# negative log-likelihood of a Bernoulli label (from MLE) -- punishes confident-and-wrong hard
def log_loss(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)  # guards log() against log(0)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def train_logistic_regression(X, y, lr=0.1, n_iters=1000):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)   # start with no opinion, all weights zero
    b = 0.0
    losses = []

    for i in range(n_iters):
        z = X @ w + b               # the linear part: w·x + b, for every row at once
        p = sigmoid(z)               # squash into probabilities

        loss = log_loss(y, p)
        losses.append(loss)

        # (p-y)*x is the derived per-point gradient; dividing by n_samples averages
        # instead of sums, so the update size doesn't depend on how many rows we have
        dw = (X.T @ (p - y)) / n_samples
        db = np.mean(p - y)          # same gradient, but for the bias (x is implicitly 1 here)

        w -= lr * dw                 # step opposite the gradient, that's what makes loss go down
        b -= lr * db

    return w, b, losses

w, b, losses = train_logistic_regression(X, y, lr=0.1, n_iters=1000)

plt.plot(losses)
plt.xlabel("iteration")
plt.ylabel("loss")
plt.title("training loss over time")
plt.show()

print("learned w:", w)
print("learned b:", b)


**2. Multinomial logistic regression**

K classes → one weight vector per class (unlike binary logreg's single vector,
since p(other classes) isn't just 1 - p(one class) anymore).

- Score:   z = W·x            (one raw score per class)
- Predict: p = softmax(z) = exp(z_k) / Σ exp(z_j)   → probabilities summing to 1
- Loss:    cross-entropy = -Σ y_k·log(p_k)          (y = one-hot true label)
- Gradient (closed form for softmax + cross-entropy): dL/dz = p - y
- Update:  W = W - lr · (p - y)ᵀ·x, repeated up to max_iter times, or until it converges

Shapes here: X (1400, 2000) · Wᵀ (2000, 10) → Z (1400, 10) → softmax row-wise → P (1400, 10)

**3. Linear Regression**

Linear Regression models the relationship between a dependent continuous target variable $y$ and one or more independent predictor features $X$ by fitting a linear equation to observed data.

- Simple Linear Regression (1 Feature):$$y = \beta_0 + \beta_1 x + \epsilon$$ 
- Multiple Linear Regression ($p$ Features):$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p + \epsilon$$
- In matrix vector notation for $N$ observations:$$Y = X\beta + \epsilon$$

Where:$Y \in \mathbb{R}^{N \times 1}$: Target vector.$X \in \mathbb{R}^{N \times (p+1)}$: Feature matrix (with a column of $1$ s inserted for the intercept $\beta_0$).$\beta \in \mathbb{R}^{(p+1) \times 1}$: Parameter weights vector.$\epsilon \in \mathbb{R}^{N \times 1}$: Unobserved residual errors.

For Ordinary Least Squares (OLS) to be the BLUE (Best Linear Unbiased Estimator), several classical statistical assumptions must hold.

1. Linearity - Y has a linear relationship with X. Plot Residuals vs. Fitted Values ($\hat{y}$). If points form a non-linear curve (e.g., U-shape), linearity is violated.

2. Independence - Residuals are uncorrelated across samples. Residuals are un-correlated with one another ($\text{Cov}(\epsilon_i, \epsilon_j) = 0$ for $i \neq j$). Plot Residuals over time / sequence order. Statistical Test: Durbin-Watson Test (Values near $2.0$ mean no autocorrelation; values $< 1.5$ indicate positive autocorrelation).

3. Homoscedasticity - Variance of residuals is constant across X. The variance of residual errors $\text{Var}(\epsilon_i) = \sigma^2$ is constant across all predicted values $\hat{y}_i$. Plot Residuals vs. Fitted Values. If the spread expands/contracts (a "funnel" or "cone" shape), the error is heteroscedastic. Statistical Tests: Breusch-Pagan Test or White Test ($p < 0.05$ indicates heteroscedasticity).

4. Normality - Residuals follow N(0, σ²) distribution. Residual errors $\epsilon$ are normally distributed around zero: $\epsilon \sim \mathcal{N}(0, \sigma^2)$.How to Detect:Q-Q Plot (Quantile-Quantile Plot): Points should lie tightly along the diagonal reference line.Statistical Tests: Shapiro-Wilk Test or Jarque-Bera Test.

5. No Multicollinearity - Features are not highly correlated. Independent features are not highly linearly correlated with each other (matrix $X^T X$ must be invertible).How to Detect:Variance Inflation Factor (VIF):$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$ (where $R_j^2$ is the coefficient of determination from regressing feature $x_j$ against all other features)Interpretation: $\text{VIF} > 5$ indicates high correlation; $\text{VIF} > 10$ indicates severe multicollinearity.

**3.1 Ordinary Least Squares (OLS) Solution**

OLS finds the parameter weights $\beta$ that minimize the Sum of Squared Residuals (SSR) analytically.Objective Loss Function $$\text{SSR}(\beta) = \sum_{i=1}^N (y_i - \mathbf{x}_i^T \beta)^2 = (Y - X\beta)^T (Y - X\beta)$$

Derivation of Normal EquationTo find the optimal weights $\hat{\beta}$, take the derivative of SSR with respect to $\beta$ and set it to $0$:

Expand loss function:$$\text{SSR}(\beta) = Y^T Y - 2\beta^T X^T Y + \beta^T X^T X \beta$$ Take gradient with respect to $\beta$:$$\frac{\partial \text{SSR}}{\partial \beta} = -2 X^T Y + 2 X^T X \beta = 0$$ Rearrange to get the Normal Equation:$$X^T X \beta = X^T Y$$ $$\mathbf{\hat{\beta} = (X^T X)^{-1} X^T Y}$$

Cons: Matrix inversion $(X^T X)^{-1}$ takes $O(p^3)$ computation time. Extremely slow or computationally intractable when $p$ (number of features) is large ($> 10,000$).

**3.2 Gradient Descent Optimization Steps :**
       
1. Initialize Weights (β) randomly
2. Compute Predictions (ŷ = Xβ)
3. Calculate Loss (MSE)
4. Compute Gradient (∂MSE / ∂β)
5. Update Weights (β := β - α * Gradient)
6. Repeat until convergence

Objective Loss Function (Mean Squared Error - MSE) $$J(\beta) = \frac{1}{2N} \sum_{i=1}^N (\mathbf{x}_i^T \beta - y_i)^2 = \frac{1}{2N} (X\beta - Y)^T (X\beta - Y)$$ (The factor $\frac{1}{2}$ cancels out neatly during differentiation)Gradient Derivation $$\nabla_{\beta} J(\beta) = \frac{1}{N} X^T (X\beta - Y) = \frac{1}{N} X^T (\hat{Y} - Y)$$ Weight Update Rule$$\beta^{(t+1)} = \beta^{(t)} - \alpha \cdot \nabla_{\beta} J(\beta)$$ $$\beta^{(t+1)} = \beta^{(t)} - \frac{\alpha}{N} X^T (\hat{Y} - Y)$$(where $\alpha$ is the Learning Rate)

Variants of Gradient Descent - 
1. Batch Gradient Descent: Uses the entire dataset $N$ to compute gradients at every step.Trait: Smooth convergence, but memory-intensive for huge datasets.
2. Stochastic Gradient Descent (SGD): Uses a single random sample ($N=1$) per update step.Trait: Extremely fast, noisy updates; can escape local minima
3. Mini-Batch Gradient Descent: Uses small batches (e.g., 32, 64, 256 samples) per update step.Trait: Industry standard; balances GPU vectorized computation with optimization stability.

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4])
y = np.array([15, 25, 35, 40])

xbar, ybar = x.mean(), y.mean()
slope = np.sum((x - xbar) * (y - ybar)) / np.sum((x - xbar) ** 2)
intercept = ybar - slope * xbar

print(f"slope={slope}, intercept={intercept}")
print("predictions:", slope * x + intercept)

# normal equation, generalizes to any number of features
X_design = np.column_stack([np.ones(len(x)), x])  # add intercept column
w = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y
print("\nnormal equation [intercept, slope]:", w)

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(x.reshape(-1, 1), y)
print("sklearn slope:", model.coef_, "intercept:", model.intercept_)

# gradient descent from scratch, should converge to the same slope/intercept
w, b = 0.0, 0.0
lr = 0.01
for step in range(2000):
    pred = w * x + b
    error = y - pred
    grad_w = -2 * np.mean(error * x)
    grad_b = -2 * np.mean(error)
    w -= lr * grad_w
    b -= lr * grad_b

print("gradient descent result: slope={:.3f}, intercept={:.3f}".format(w, b))

**4. Support Vector Machine (SVM)**- works by finding an optimal decision boundary (hyperplane) that maximizes the margin between different classes.

Hyperplane: The decision boundary separating classes. In an $n$-dimensional space, a hyperplane is an $(n-1)$-dimensional subspace defined as $w^T x + b = 0$.

Support Vectors: The data points closest to the hyperplane. These are the critical points that define the position and orientation of the margin. Removing other data points changes nothing; removing a support vector shifts the boundary.

Margin: The distance between the hyperplane and the closest support vector on either side. SVM maximizes this distance ($2 / \Vert{}w\Vert{}$).

Soft Margin & C Hyperparameter: Real-world data is rarely linearly separable. SVM uses a slack variable ($\xi_i$) to allow some misclassifications:$$\min_{w, b, \xi} \frac{1}{2} \Vert{}w\Vert{}^2 + C \sum_{i=1}^N \xi_i$$
    - High $C$: Low tolerance for misclassifications $\rightarrow$ narrower margin $\rightarrow$ risks overfitting.
    - Low $C$: High tolerance for misclassifications $\rightarrow$ wider margin $\rightarrow$ risks underfitting.

Kernel Trick: Transforms non-linearly separable data in low dimensions into a higher-dimensional space where it becomes linearly separable—without explicitly computing coordinates in that high-dimensional space.Common Kernels: Linear, Polynomial, RBF (Radial Basis Function / Gaussian), Sigmoid.